### Entradas
- multiplicadores dos parâmetros
- OF Value
- Simulation

### Saídas
- descritores globais do alvo
- descritores verticais resumidos
- descritores do modelo completo
- descritores em poços

### Leituras estatísticas
- Spearman
- Regressão linear / Lasso
- Random Forest
- PLS
- PCA no espaço das respostas

In [3]:

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import importlib.util

BASE_DIR = Path("")
CALIBRATION_FILE = BASE_DIR / "Uncertain_parameters_and_OF_values_sens1.xlsx"
MASTER_TABLE_FILE = BASE_DIR / "master_table.parquet"

spec = importlib.util.spec_from_file_location("master_table_config", BASE_DIR / "master_table_config.py")
cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cfg)

cfg.expanded_master_table_columns()[:15]


['Simulation_ID',
 'Simulation',
 'OutputPath',
 'OF Value',
 'Carbo_GrainsProdvsTime',
 'Carbo_MudProdvsTime',
 'Carbo_RudProdvsTime',
 'LutitesProdvsTime',
 'S1Supply0',
 'S2Supply0',
 'target_volume',
 'target_fraction_global',
 'n_clusters',
 'connected_fraction',
 'percolation_x']

In [4]:

# 1) Carregar tabela de calibração

calib_df = pd.read_excel(CALIBRATION_FILE)
calib_df = calib_df.rename(columns={"Unnamed: 0": "Simulation_ID"})
calib_df.head()


,Simulation_ID,Carbo_GrainsProdvsTime,Carbo_GrainsProdvsBat,Carbo_MudProdvsTime,Carbo_MudProdvsBat,Eustasy0,Subsidence124.4,Subsidence124,OutputPath,OF Value,Simulation
0,Sim0,1.420988,1.354835,1.182428,0.907213,1.278636,0.427700,1.228338,output0,41.255853,0
1,Sim1,1.487285,1.852434,1.420774,1.067469,1.529987,1.529689,1.327791,output1,58.126545,1
2,Sim2,0.985376,1.584350,0.413524,1.039806,1.493145,0.750323,1.840956,output2,36.182783,2
3,Sim3,1.296345,0.123549,0.207220,1.535241,0.351723,0.827885,1.797445,output3,43.449475,3
4,Sim4,1.198141,0.575405,0.525354,0.697009,1.057621,0.493306,1.835604,output4,22.092793,4



## Descritores escolhidos nesta primeira versão

### Núcleo global do alvo
- `target_volume`
- `target_fraction_global`
- `n_clusters`
- `connected_fraction`
- `percolation_x`, `percolation_y`, `percolation_z`

### Núcleo vertical
- `Ttot`
- `target_fraction_col`
- `n_packages`
- `Tpack_max`
- `ICV_env`
- `Tgap_sum`

### Modelo de fácies completo
- `facies_entropy_global`
- `facies_diversity_global`
- `n_facies_per_column_mean`

### Poços
- `well_score_mean`
- `well_score_min`
- `well_score_std`

Para os descritores tipo mapa, vamos salvar resumos:
`mean`, `std`, `p10`, `p50`, `p90`.


In [5]:

# 2) Funções utilitárias para resumir descritores de mapa

def summarize_array(values, prefix):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return {f"{prefix}_mean": np.nan,
                f"{prefix}_std": np.nan,
                f"{prefix}_p10": np.nan,
                f"{prefix}_p50": np.nan,
                f"{prefix}_p90": np.nan}
    return {
        f"{prefix}_mean": float(np.mean(values)),
        f"{prefix}_std": float(np.std(values)),
        f"{prefix}_p10": float(np.percentile(values, 10)),
        f"{prefix}_p50": float(np.percentile(values, 50)),
        f"{prefix}_p90": float(np.percentile(values, 90)),
    }

summarize_array([1,2,3,4,5], "Ttot")


{'Ttot_mean': 3.0,
 'Ttot_std': 1.4142135623730951,
 'Ttot_p10': 1.4,
 'Ttot_p50': 3.0,
 'Ttot_p90': 4.6}


## Extrator de descritores

Esta é a célula que você vai conectar ao seu software.

A ideia é:

- abrir o grid da simulação
- calcular os descritores
- devolver um dicionário com números

Por enquanto, a função está como **template**.


In [ ]:

def extract_model_descriptors(
    simulation_row: pd.Series,
    target_name: str = "selected_target",
    base_model_path: str | None = None,
    wells_data: object | None = None,
):
    """
    Template.

    Substituir o corpo desta função por chamadas ao seu software:
    - carregar grid da simulação
    - calcular descritores globais
    - calcular mapas verticais
    - resumir mapas
    - comparar com base (se quiser)
    - calcular métricas de poços (se houver)

    Esperado: retornar um dict com valores escalares.
    """
    # EXEMPLO de estrutura de saída:
    result = {
        "target_volume": np.nan,
        "target_fraction_global": np.nan,
        "n_clusters": np.nan,
        "connected_fraction": np.nan,
        "percolation_x": np.nan,
        "percolation_y": np.nan,
        "percolation_z": np.nan,
        "facies_entropy_global": np.nan,
        "facies_diversity_global": np.nan,
        "n_facies_per_column_mean": np.nan,
        "well_score_mean": np.nan,
        "well_score_min": np.nan,
        "well_score_std": np.nan,
    }



    return result


In [7]:

# 3) Montar a master table
#
# Nesta primeira versão, o loop está pronto e a parte do extrator
# precisa ser conectada ao seu software.

master_rows = []

for _, row in calib_df.iterrows():
    base_info = {
        "Simulation_ID": row.get("Simulation_ID"),
        "Simulation": row.get("Simulation"),
        "OutputPath": row.get("OutputPath"),
        "OF Value": row.get("OF Value"),
    }
    for p in cfg.PARAMETER_COLUMNS:
        base_info[p] = row.get(p)

    desc = extract_model_descriptors(row)
    base_info.update(desc)
    master_rows.append(base_info)

master_df = pd.DataFrame(master_rows)
master_df.head()


,Simulation_ID,Simulation,OutputPath,OF Value,Carbo_GrainsProdvsTime,Carbo_MudProdvsTime,Carbo_RudProdvsTime,LutitesProdvsTime,S1Supply0,S2Supply0,...,connected_fraction,percolation_x,percolation_y,percolation_z,facies_entropy_global,facies_diversity_global,n_facies_per_column_mean,well_score_mean,well_score_min,well_score_std
0,Sim0,0,output0,41.255853,1.420988,1.182428,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sim1,1,output1,58.126545,1.487285,1.420774,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sim2,2,output2,36.182783,0.985376,0.413524,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sim3,3,output3,43.449475,1.296345,0.207220,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sim4,4,output4,22.092793,1.198141,0.525354,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:

# Salvar master table
# Quando o extrator estiver conectado, descomente:

# master_df.to_parquet(MASTER_TABLE_FILE, index=False)
# master_df.to_csv(BASE_DIR / "master_table.csv", index=False)



## Se você já tiver a master table pronta
Basta carregar daqui em diante.


In [ ]:

# Use esta célula depois que a master table estiver preenchida de verdade.
# master_df = pd.read_parquet(MASTER_TABLE_FILE)



# 4) Escolher X e Y

### X = entradas
Parâmetros da calibração

### Y = saídas
Descritores do modelo


In [ ]:

X_cols = cfg.PARAMETER_COLUMNS

Y_cols = [
    "target_volume",
    "target_fraction_global",
    "n_clusters",
    "connected_fraction",
    "percolation_x",
    "percolation_y",
    "percolation_z",
    "Ttot_mean",
    "target_fraction_col_mean",
    "n_packages_mean",
    "Tpack_max_mean",
    "ICV_env_mean",
    "Tgap_sum_mean",
    "facies_entropy_global",
    "facies_diversity_global",
    "n_facies_per_column_mean",
    "well_score_mean",
]

available_Y_cols = [c for c in Y_cols if c in master_df.columns]
master_df[X_cols + available_Y_cols].head()



# 5) Correlação de Spearman

Primeira leitura:
- sinal positivo / negativo
- força da associação monotônica
- triagem inicial de parâmetros mais influentes


In [ ]:

if len(available_Y_cols) > 0:
    corr = pd.DataFrame(index=X_cols, columns=available_Y_cols, dtype=float)

    for x in X_cols:
        for y in available_Y_cols:
            valid = master_df[[x, y]].dropna()
            if len(valid) >= 3:
                corr.loc[x, y] = spearmanr(valid[x], valid[y]).statistic
            else:
                corr.loc[x, y] = np.nan

    plt.figure(figsize=(1.0 * len(available_Y_cols) + 4, 6))
    im = plt.imshow(corr.astype(float), aspect="auto")
    plt.colorbar(im, label="Spearman")
    plt.xticks(range(len(available_Y_cols)), available_Y_cols, rotation=90)
    plt.yticks(range(len(X_cols)), X_cols)
    plt.title("Parâmetro × descritor — correlação de Spearman")
    plt.tight_layout()
    plt.show()

corr



# 6) Regressão linear e Lasso

Use para:
- efeito médio de primeira ordem
- sinal dos coeficientes
- seleção de variáveis


In [ ]:

def fit_linear_and_lasso(df, x_cols, y_col):
    data = df[x_cols + [y_col]].dropna()
    if len(data) < 8:
        raise ValueError(f"Poucos dados válidos para {y_col}")

    X = data[x_cols]
    y = data[y_col]

    lin_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])
    lin_pipe.fit(X, y)
    lin_coef = pd.Series(lin_pipe.named_steps["model"].coef_, index=x_cols, name="Linear")

    lasso_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LassoCV(cv=min(5, len(X)), random_state=42))
    ])
    lasso_pipe.fit(X, y)
    lasso_coef = pd.Series(lasso_pipe.named_steps["model"].coef_, index=x_cols, name="Lasso")

    return pd.concat([lin_coef, lasso_coef], axis=1)

if "Ttot_mean" in master_df.columns:
    coef_df = fit_linear_and_lasso(master_df, X_cols, "Ttot_mean")
    coef_df.plot(kind="bar", figsize=(10, 5))
    plt.title("Coeficientes — Ttot_mean")
    plt.ylabel("Coeficiente")
    plt.tight_layout()
    plt.show()

    coef_df



# 7) Random Forest + permutation importance

Use para:
- capturar não linearidade
- capturar interações implícitas
- ranquear importância preditiva


In [ ]:

def fit_random_forest_with_importance(df, x_cols, y_col, n_estimators=300):
    data = df[x_cols + [y_col]].dropna()
    if len(data) < 10:
        raise ValueError(f"Poucos dados válidos para {y_col}")

    X = data[x_cols]
    y = data[y_col]

    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=42,
        min_samples_leaf=2
    )
    rf.fit(X, y)

    perm = permutation_importance(rf, X, y, n_repeats=20, random_state=42)
    imp = pd.Series(perm.importances_mean, index=x_cols).sort_values(ascending=False)

    return rf, imp, X, y

if "connected_fraction" in master_df.columns:
    rf_model, rf_imp, X_rf, y_rf = fit_random_forest_with_importance(master_df, X_cols, "connected_fraction")
    rf_imp.plot(kind="bar", figsize=(8, 4))
    plt.title("Permutation importance — connected_fraction")
    plt.ylabel("Importância")
    plt.tight_layout()
    plt.show()

    rf_imp



# 8) PDP / efeito visual de um parâmetro

Use quando a Random Forest indicar que um parâmetro é importante.


In [ ]:

if "connected_fraction" in master_df.columns:
    top_features = list(rf_imp.index[:2])
    fig, ax = plt.subplots(figsize=(8, 4))
    PartialDependenceDisplay.from_estimator(
        rf_model,
        X_rf,
        features=[top_features[0]],
        ax=ax
    )
    plt.tight_layout()
    plt.show()



# 9) PLS Regression — bloco de entradas vs bloco de saídas

Use para enxergar a ligação entre:
- espaço dos parâmetros
- espaço das respostas


In [ ]:

if len(available_Y_cols) >= 2:
    pls_data = master_df[X_cols + available_Y_cols].dropna()
    if len(pls_data) >= 8:
        X_pls = StandardScaler().fit_transform(pls_data[X_cols])
        Y_pls = StandardScaler().fit_transform(pls_data[available_Y_cols])

        pls = PLSRegression(n_components=min(2, len(X_cols), len(available_Y_cols)))
        pls.fit(X_pls, Y_pls)

        x_weights = pd.Series(pls.x_weights_[:, 0], index=X_cols).sort_values(key=np.abs, ascending=False)
        y_weights = pd.Series(pls.y_weights_[:, 0], index=available_Y_cols).sort_values(key=np.abs, ascending=False)

        display(pd.DataFrame({"x_weight_comp1": x_weights}))
        display(pd.DataFrame({"y_weight_comp1": y_weights}))



# 10) PCA no espaço das respostas

Use para enxergar:
- diversidade do ensemble
- famílias de comportamento
- agrupamentos no espaço das saídas


In [ ]:

if len(available_Y_cols) >= 3:
    response_data = master_df[available_Y_cols].dropna()
    if len(response_data) >= 8:
        Y_scaled = StandardScaler().fit_transform(response_data)
        pca = PCA(n_components=2, random_state=42)
        Y_pca = pca.fit_transform(Y_scaled)

        plt.figure(figsize=(7, 5))
        plt.scatter(Y_pca[:, 0], Y_pca[:, 1], alpha=0.8)
        plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
        plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
        plt.title("PCA do espaço das respostas")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()



# 11) Próximos passos

Quando a master table estiver preenchida de verdade, o caminho natural é:

1. rodar Spearman para triagem rápida  
2. rodar regressão linear / Lasso nos descritores principais  
3. rodar Random Forest para detectar não linearidade e interação  
4. rodar PLS para enxergar relação entre blocos  
5. rodar PCA / clustering no espaço das respostas  
6. depois entrar em DGSA

Ou seja:
**DGSA não é o primeiro passo.**
Ele entra quando o conjunto de descritores já estiver estável.
